In [0]:
from pyspark.sql import functions as F, Window

#to read a bronze table using its name
CATALOG = "brazilian_ecommerce_dataset_olist"
SCHEMA = "brazilian_ecommerce_dataset_olist_schema"
def bronze(table):
    return spark.table(f"{CATALOG}.{SCHEMA}.bronze_{table}")

#to save processed data as a Delta table in the Silver layer
def save(df, table):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_{table}") 

orders = bronze("orders").select("order_id", "customer_id", "order_status", 
                                 *[F.to_timestamp(c).alias(c) for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]]) 

counts = orders.groupBy("order_id").count()
# counts.show()
#to see if an order id how many time it is repeated 
orders = orders.join(counts.withColumnRenamed("count", "_id_count"), "order_id", "left")
valid = (F.col("order_id").isNotNull() & F.col("customer_id").isNotNull() &
         F.col("order_purchase_timestamp").isNotNull() & F.col("order_estimated_delivery_date").isNotNull() &
         (F.col("_id_count") == 1) &
         (F.col("order_estimated_delivery_date") >= F.col("order_purchase_timestamp")) &
         (F.col("order_delivered_customer_date").isNull() |
          (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))))

save(orders.filter(~valid), "invalid_orders")
orders = orders.filter(valid).drop("_id_count") #keep only valid orders

#to have each customer's location --> customer's state
customers = bronze("customers").select("customer_id", "customer_state").dropDuplicates(["customer_id"])

# orders = the purchase transaction as a whole
# order_items = the individual items included in that transaction (when same order_id order the same item (product_id) multiple times this value increases to that number)
# products = information describing each product
items = (bronze("order_items").withColumn("price", F.col("price").cast("double"))
         .withColumn("freight_value", F.col("freight_value").cast("double"))
         .withColumn("order_item_id", F.col("order_item_id").cast("int"))) 
         
# Items are one-to-many per order --> Aggregate before joining to prevent KPI inflation
item_summary = (items.groupBy("order_id")
               .agg(F.count("*").alias("order_item_count"),
                    F.sum("price").alias("order_price"),
                    F.sum("freight_value").alias("order_freight_value"),
                    F.sum(F.col("price") + F.col("freight_value")).alias("order_total_value")))

# Reviews can be multiple per order --> average score per order, not a duplicate producing join
reviews = (bronze("reviews").withColumn("review_score", F.col("review_score").try_cast("double")).groupBy("order_id").agg(F.avg("review_score").alias("review_score"), F.count("*").alias("review_count")))

products = bronze("products").select("product_id", "product_category_name").dropDuplicates(["product_id"])
#how many items from each product category appear in each order.
category_counts = (items.join(products, "product_id", "left")
                   .groupBy("order_id", "product_category_name").agg(F.count("*").alias("category_items")))

w = Window.partitionBy("order_id").orderBy(F.desc("category_items"), F.asc_nulls_last("product_category_name"))
#to keep the category that has the highest number of order in that order id
primary_category = (category_counts.withColumn("_rank", F.row_number().over(w)).filter("_rank = 1").select("order_id", F.col("product_category_name").alias("primary_category")))

translations = (bronze("category_translation")
                .select("product_category_name", "product_category_name_english")
                .dropDuplicates(["product_category_name"]))
primary_category = primary_category.join(translations, primary_category.primary_category == translations.product_category_name, "left").drop("product_category_name")
result = (orders.join(customers, "customer_id", "left")
          .join(item_summary, "order_id", "left")
          .join(reviews, "order_id", "left")
          .join(primary_category, "order_id", "left")
          .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
          .withColumn("delivery_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp"))/86400.0)
          .withColumn("delay_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_estimated_delivery_date"))/86400.0)
          .withColumn("is_late", F.when((F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNotNull(), F.when(F.col("delay_days") > 0, 1).otherwise(0)).otherwise(F.lit(None).cast("int")))
          .withColumn("is_delivered", (F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNotNull()))
save(result, "orders_enriched")
assert result.select("order_id").distinct().count() == result.count(), "Order grain not unique"
print("Silver rows:", result.count(), "Invalid order rows:", spark.table(f"{CATALOG}.{SCHEMA}.silver_invalid_orders").count())
